# Helicone Token 追蹤實戰

**課程**: Session 1 - Technical Foundations  
**目標**: 學會使用 Helicone 追蹤 LLM API Token 使用狀態和成本

---

## 這個 Lab 你會學到

- 如何設定 Helicone 追蹤
- 使用 LLM 生成 function 實作
- 透過 Unit Test 驗證 LLM 生成的程式碼
- 使用 Helicone API 分析使用資料

---

## Lab 結構

- **Part 0**: Helicone 設定
- **Part 1**: Function 實作練習（使用 LLM 生成）
- **Part 2**: Helicone API 分析

---

## Part 0: Helicone 設定

### 目標

設定 Helicone token 追蹤，自動記錄所有 API 使用數據

---

**Helicone** 是一個 LLM 監控工具，可以追蹤:
- Token 使用量（input/output tokens）
- API 呼叫成本
- API 呼叫記錄
- 回應時間
- 錯誤率

### Task 0.1: 設定 API Keys

**需要準備的 API Keys**:
1. **Google Gemini API Key**: 從 [Google AI Studio](https://aistudio.google.com/app/apikey) 取得
2. **Helicone API Key**: 使用課程提供的 Helicone API Key

In [ ]:
import os

# Google Colab Secrets
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
HELICONE_API_KEY = userdata.get("HELICONE_API_KEY")

print("API Keys 設定完成")

### Task 0.2: 設定環境變數

In [ ]:
# 你的名字（用於在 Helicone 中識別）
USER_NAME = "your-name"

# 設定環境變數
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["HELICONE_API_KEY"] = HELICONE_API_KEY

print(f"使用者: {USER_NAME}")

### Task 0.3: 建立 Helicone Client

透過設定 Helicone 的 gateway headers，所有的 API 請求都會自動被追蹤

In [ ]:
import requests
import json

# Helicone 設定
HELICONE_GATEWAY_URL = "https://gateway.helicone.ai"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com"

HELICONE_HEADERS = {
    "Helicone-Auth": f"Bearer {HELICONE_API_KEY}",
    "Helicone-Target-URL": GEMINI_BASE_URL,
    "Helicone-User-Id": USER_NAME,
    "Content-Type": "application/json",
}


def call_gemini_with_helicone(
    prompt, model_name="gemini-2.0-flash", max_tokens=1024, extra_properties=None
):
    """
    透過 Helicone 代理呼叫 Gemini API

    Args:
        prompt: 使用者的提示詞
        model_name: 模型名稱
        max_tokens: 最大輸出 tokens
        extra_properties: 額外的 Helicone 屬性（用於分類）

    Returns:
        dict: 包含回應文字、token 使用量、成本的字典
    """
    headers = HELICONE_HEADERS.copy()

    if extra_properties:
        for key, value in extra_properties.items():
            headers[f"Helicone-Property-{key}"] = value

    data = {
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"maxOutputTokens": max_tokens},
    }

    url = f"{HELICONE_GATEWAY_URL}/v1beta/models/{model_name}:generateContent?key={GOOGLE_API_KEY}"
    response = requests.post(url, headers=headers, json=data)

    if response.status_code != 200:
        raise Exception(f"API 請求失敗: {response.status_code} - {response.text}")

    result = response.json()
    text = result["candidates"][0]["content"]["parts"][0]["text"]

    usage_metadata = result.get("usageMetadata", {})
    input_tokens = usage_metadata.get("promptTokenCount", 0)
    output_tokens = usage_metadata.get("candidatesTokenCount", 0)
    total_tokens = usage_metadata.get("totalTokenCount", input_tokens + output_tokens)

    cost_usd = 0.0
    helicone_cost = response.headers.get("Helicone-Cost-USD")
    if helicone_cost:
        try:
            cost_usd = float(helicone_cost)
        except (ValueError, TypeError):
            cost_usd = 0.0

    return {
        "text": text,
        "usage": {
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": total_tokens,
        },
        "cost_usd": cost_usd,
    }


print("Helicone 設定完成")

### Task 0.4: 測試連線

In [ ]:
try:
    response = call_gemini_with_helicone(
        prompt=f"Hi! I'm {USER_NAME}.",
        max_tokens=50,
        extra_properties={"Task": "Connection-Test"},
    )

    print("連線測試成功")
    print(f"\nGemini 回應: {response['text']}")
    print(f"\nToken 使用:")
    print(f"  Input: {response['usage']['input_tokens']}")
    print(f"  Output: {response['usage']['output_tokens']}")
    print(f"  Total: {response['usage']['total_tokens']}")

except Exception as e:
    print(f"連線失敗: {str(e)}")

---

## Part 1: Function 實作練習

### 目標

使用 LLM 生成 function 實作，並透過 Unit Test 驗證正確性

### 練習流程

1. 我們提供 function signature 和 docstring
2. 你把它給 LLM，請 LLM 生成實作
3. 貼上 LLM 生成的程式碼
4. 執行 Unit Test 驗證

---

### 背景：書籍管理系統

我們要實作一個書籍管理系統的核心 functions：

| Function | 說明 |
|----------|------|
| `validate_book_data` | 驗證書籍資料是否完整 |
| `create_book` | 新增書籍到資料庫 |
| `search_books` | 多條件動態搜尋書籍 |

**資料庫 Schema**:
```sql
CREATE TABLE books (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    author TEXT NOT NULL,
    isbn TEXT UNIQUE NOT NULL,
    published_year INTEGER,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
```

### 預先設定：資料庫連線

以下是我們提供的資料庫設定，你不需要修改：

In [ ]:
import sqlite3

DATABASE = ":memory:"  # 使用記憶體資料庫，方便測試


def get_db_connection():
    """取得資料庫連線"""
    conn = sqlite3.connect(DATABASE)
    conn.row_factory = sqlite3.Row
    return conn


def init_db(conn):
    """初始化資料庫"""
    conn.execute(
        """
        CREATE TABLE IF NOT EXISTS books (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT NOT NULL,
            author TEXT NOT NULL,
            isbn TEXT UNIQUE NOT NULL,
            published_year INTEGER,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """
    )
    conn.commit()


print("資料庫設定完成")

---

### 練習 1.1: 實作 `validate_book_data`

**任務**: 請用 LLM 生成這個 function 的實作

**步驟**:
1. 執行下方 cell，用 API 生成實作
2. 複製 LLM 回傳的程式碼
3. 貼到「TODO」區塊中，取代 `raise NotImplementedError`
4. 執行測試驗證

In [ ]:
# Prompt 設計

PROMPT_VALIDATE = """
請實作這個 Python function 的內容（只需要 function body）：

def validate_book_data(data: dict) -> tuple[bool, str | None]:
    \"\"\"
    驗證書籍資料是否完整

    檢查規則：
    - title: 必填，不可為空字串
    - author: 必填，不可為空字串
    - isbn: 必填，不可為空字串
    - published_year: 選填，如果有值必須是正整數

    Args:
        data: 包含書籍資料的字典

    Returns:
        tuple: (is_valid, error_message)
            - 驗證通過: (True, None)
            - 驗證失敗: (False, "錯誤訊息")

    Examples:
        >>> validate_book_data({"title": "Book", "author": "Author", "isbn": "123"})
        (True, None)

        >>> validate_book_data({"title": "", "author": "Author", "isbn": "123"})
        (False, "title is required")

        >>> validate_book_data({"title": "Book", "author": "Author", "isbn": "123", "published_year": -1})
        (False, "published_year must be a positive integer")
    \"\"\"

只需要回傳 function body 的程式碼，不要包含 def 和 docstring。
"""

print("Prompt 已準備，執行下一個 cell 來生成實作")

In [ ]:
# 用 API 生成實作

response = call_gemini_with_helicone(
    prompt=PROMPT_VALIDATE,
    max_tokens=500,
    extra_properties={"Task": "Generate-validate_book_data"},
)

print("LLM 生成的實作：")
print("=" * 50)
print(response["text"])
print("=" * 50)
print(f"\nToken 使用: {response['usage']['total_tokens']}")

In [ ]:
def validate_book_data(data: dict) -> tuple[bool, str | None]:
    """
    驗證書籍資料是否完整

    檢查規則：
    - title: 必填，不可為空字串
    - author: 必填，不可為空字串
    - isbn: 必填，不可為空字串
    - published_year: 選填，如果有值必須是正整數
    """
    # ========================================
    # TODO: 貼上 LLM 生成的實作
    # ========================================
    raise NotImplementedError("請貼上 LLM 生成的實作")


print("validate_book_data 已定義")

In [ ]:
# 測試 validate_book_data

import unittest


class TestValidateBookData(unittest.TestCase):

    def test_valid_data_all_fields(self):
        """完整資料應該通過驗證"""
        data = {
            "title": "Python 入門",
            "author": "王小明",
            "isbn": "978-123-456",
            "published_year": 2024,
        }
        is_valid, error = validate_book_data(data)
        self.assertTrue(is_valid)
        self.assertIsNone(error)

    def test_valid_data_required_only(self):
        """只有必填欄位也應該通過"""
        data = {"title": "Book", "author": "Author", "isbn": "123"}
        is_valid, error = validate_book_data(data)
        self.assertTrue(is_valid)
        self.assertIsNone(error)

    def test_missing_title(self):
        """缺少 title 應該失敗"""
        data = {"author": "Author", "isbn": "123"}
        is_valid, error = validate_book_data(data)
        self.assertFalse(is_valid)
        self.assertIsNotNone(error)

    def test_empty_title(self):
        """空字串 title 應該失敗"""
        data = {"title": "", "author": "Author", "isbn": "123"}
        is_valid, error = validate_book_data(data)
        self.assertFalse(is_valid)
        self.assertIsNotNone(error)

    def test_missing_author(self):
        """缺少 author 應該失敗"""
        data = {"title": "Book", "isbn": "123"}
        is_valid, error = validate_book_data(data)
        self.assertFalse(is_valid)
        self.assertIsNotNone(error)

    def test_missing_isbn(self):
        """缺少 isbn 應該失敗"""
        data = {"title": "Book", "author": "Author"}
        is_valid, error = validate_book_data(data)
        self.assertFalse(is_valid)
        self.assertIsNotNone(error)

    def test_invalid_published_year_negative(self):
        """負數 published_year 應該失敗"""
        data = {
            "title": "Book",
            "author": "Author",
            "isbn": "123",
            "published_year": -1,
        }
        is_valid, error = validate_book_data(data)
        self.assertFalse(is_valid)
        self.assertIsNotNone(error)

    def test_invalid_published_year_zero(self):
        """0 的 published_year 應該失敗"""
        data = {
            "title": "Book",
            "author": "Author",
            "isbn": "123",
            "published_year": 0,
        }
        is_valid, error = validate_book_data(data)
        self.assertFalse(is_valid)
        self.assertIsNotNone(error)


# 執行測試
suite = unittest.TestLoader().loadTestsFromTestCase(TestValidateBookData)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

passed = result.testsRun - len(result.failures) - len(result.errors)
print(f"\n結果: {passed}/{result.testsRun} 測試通過")

---

### 練習 1.2: 實作 `create_book`

**任務**: 根據 PM 給的需求規格，自己設計 prompt 讓 LLM 生成實作

**PM 需求規格**:

```
Function: create_book
功能：新增一本書到資料庫

輸入：
- conn: 資料庫連線
- data: dict，包含 title, author, isbn (必填), published_year (選填)

輸出：
- 成功：回傳新增的書籍資料 (dict)，要包含自動產生的 id
- 失敗：回傳 None（例如 ISBN 重複時）

技術要求：
- 使用 parameterized query
- 處理 sqlite3.IntegrityError
```

**提示**: 參考練習 1.1 的 prompt 結構，思考如何讓 LLM 理解你的需求

In [ ]:
# TODO: 根據 PM 需求規格，設計你的 prompt

PROMPT_CREATE = """
# ========================================
# 在這裡寫你的 prompt
# ========================================
# 提示：
# - 明確描述 function signature
# - 說明輸入參數和輸出格式
# - 提供資料表 schema
# - 列出注意事項（SQL injection 防護、錯誤處理等）


"""

print(f"你的 Prompt 長度: {len(PROMPT_CREATE)} 字元")
print("\n設計好後，執行下一個 cell 來生成實作")

In [ ]:
# 用 API 生成實作

response = call_gemini_with_helicone(
    prompt=PROMPT_CREATE,
    max_tokens=800,
    extra_properties={"Task": "Generate-create_book"},
)

print("LLM 生成的實作：")
print("=" * 50)
print(response["text"])
print("=" * 50)
print(f"\nToken 使用: {response['usage']['total_tokens']}")

In [ ]:
def create_book(conn, data: dict) -> dict | None:
    """
    新增書籍到資料庫

    Args:
        conn: sqlite3 資料庫連線物件
        data: 包含書籍資料的字典

    Returns:
        dict: 新增成功時回傳書籍資料（包含 id）
        None: 新增失敗時回傳 None
    """
    # ========================================
    # TODO: 貼上 LLM 生成的實作
    # ========================================
    raise NotImplementedError("請貼上 LLM 生成的實作")


print("create_book 已定義")

In [ ]:
# 測試 create_book


class TestCreateBook(unittest.TestCase):

    def setUp(self):
        self.conn = sqlite3.connect(":memory:")
        self.conn.row_factory = sqlite3.Row
        init_db(self.conn)

    def tearDown(self):
        self.conn.close()

    def test_create_book_success(self):
        """成功新增書籍"""
        data = {
            "title": "Python 入門",
            "author": "王小明",
            "isbn": "978-123-456",
            "published_year": 2024,
        }
        result = create_book(self.conn, data)

        self.assertIsNotNone(result)
        self.assertIn("id", result)
        self.assertEqual(result["title"], "Python 入門")
        self.assertEqual(result["author"], "王小明")
        self.assertEqual(result["isbn"], "978-123-456")

    def test_create_book_without_year(self):
        """不含 published_year 也能新增"""
        data = {"title": "Book", "author": "Author", "isbn": "123"}
        result = create_book(self.conn, data)

        self.assertIsNotNone(result)
        self.assertIn("id", result)

    def test_create_book_duplicate_isbn(self):
        """ISBN 重複應該回傳 None"""
        data1 = {"title": "Book 1", "author": "Author", "isbn": "SAME-ISBN"}
        data2 = {"title": "Book 2", "author": "Author", "isbn": "SAME-ISBN"}

        result1 = create_book(self.conn, data1)
        result2 = create_book(self.conn, data2)

        self.assertIsNotNone(result1)
        self.assertIsNone(result2)

    def test_create_book_auto_increment_id(self):
        """ID 應該自動遞增"""
        data1 = {"title": "Book 1", "author": "Author", "isbn": "ISBN-1"}
        data2 = {"title": "Book 2", "author": "Author", "isbn": "ISBN-2"}

        result1 = create_book(self.conn, data1)
        result2 = create_book(self.conn, data2)

        self.assertEqual(result1["id"], 1)
        self.assertEqual(result2["id"], 2)


# 執行測試
suite = unittest.TestLoader().loadTestsFromTestCase(TestCreateBook)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

passed = result.testsRun - len(result.failures) - len(result.errors)
print(f"\n結果: {passed}/{result.testsRun} 測試通過")

---

### 練習 1.3: 實作 `search_books`

**任務**: 根據 PM 給的需求規格，自己設計 prompt 讓 LLM 生成實作

**PM 需求規格**:

```
Function: search_books
功能：多條件動態搜尋書籍

輸入：
- conn: 資料庫連線
- filters: dict，所有條件都是選填
  - title: 書名關鍵字（模糊搜尋，不分大小寫）
  - author: 作者關鍵字（模糊搜尋，不分大小寫）
  - year_from: 出版年份起始
  - year_to: 出版年份結束
  - limit: 回傳筆數上限（預設 10，最大 100）
  - offset: 分頁偏移（預設 0）

輸出：
- list[dict]: 符合條件的書籍列表
- 依 id 升冪排序

技術要求：
- 動態組合 SQL WHERE 條件
- 模糊搜尋用 LIKE + %
- 大小寫不敏感用 LOWER()
- 防止 SQL injection
```

**挑戰**: 這個 function 比較複雜，需要仔細設計 prompt 才能讓 LLM 正確處理所有邊界條件

In [ ]:
# TODO: 根據 PM 需求規格，設計你的 prompt

PROMPT_SEARCH = """
# ========================================
# 在這裡寫你的 prompt
# ========================================
# 提示：
# - 這個 function 有很多邊界條件要處理
# - 想清楚每個 filter 的處理方式
# - limit 的預設值和最大值
# - 如何動態組合 WHERE 條件
# - 如何做到大小寫不敏感的模糊搜尋


"""

print(f"你的 Prompt 長度: {len(PROMPT_SEARCH)} 字元")
print("\n設計好後，執行下一個 cell 來生成實作")

In [ ]:
# 用 API 生成實作

response = call_gemini_with_helicone(
    prompt=PROMPT_SEARCH,
    max_tokens=1000,
    extra_properties={"Task": "Generate-search_books"},
)

print("LLM 生成的實作：")
print("=" * 50)
print(response["text"])
print("=" * 50)
print(f"\nToken 使用: {response['usage']['total_tokens']}")

In [ ]:
def search_books(conn, filters: dict) -> list[dict]:
    """
    多條件動態搜尋書籍

    Args:
        conn: sqlite3 資料庫連線物件
        filters: 搜尋條件字典

    Returns:
        list[dict]: 符合條件的書籍列表
    """
    # ========================================
    # TODO: 貼上 LLM 生成的實作
    # ========================================
    raise NotImplementedError("請貼上 LLM 生成的實作")


print("search_books 已定義")

In [ ]:
# 測試 search_books


class TestSearchBooks(unittest.TestCase):

    def setUp(self):
        self.conn = sqlite3.connect(":memory:")
        self.conn.row_factory = sqlite3.Row
        init_db(self.conn)

        # 新增測試資料
        test_books = [
            ("Python 入門", "王小明", "ISBN-001", 2020),
            ("Python 進階", "王小明", "ISBN-002", 2021),
            ("JavaScript 基礎", "李大華", "ISBN-003", 2019),
            ("React 實戰", "李大華", "ISBN-004", 2022),
            ("Database Design", "John Smith", "ISBN-005", 2018),
        ]
        for title, author, isbn, year in test_books:
            self.conn.execute(
                "INSERT INTO books (title, author, isbn, published_year) VALUES (?, ?, ?, ?)",
                (title, author, isbn, year),
            )
        self.conn.commit()

    def tearDown(self):
        self.conn.close()

    def test_search_no_filters(self):
        """無條件搜尋應回傳所有書籍（受 limit 限制）"""
        result = search_books(self.conn, {})
        self.assertIsInstance(result, list)
        self.assertEqual(len(result), 5)

    def test_search_by_title(self):
        """依 title 模糊搜尋"""
        result = search_books(self.conn, {"title": "python"})
        self.assertEqual(len(result), 2)
        for book in result:
            self.assertIn("python", book["title"].lower())

    def test_search_by_title_case_insensitive(self):
        """title 搜尋應不分大小寫"""
        result1 = search_books(self.conn, {"title": "PYTHON"})
        result2 = search_books(self.conn, {"title": "python"})
        self.assertEqual(len(result1), len(result2))

    def test_search_by_author(self):
        """依 author 模糊搜尋"""
        result = search_books(self.conn, {"author": "王"})
        self.assertEqual(len(result), 2)

    def test_search_by_year_range(self):
        """依年份範圍搜尋"""
        result = search_books(self.conn, {"year_from": 2020, "year_to": 2021})
        self.assertEqual(len(result), 2)
        for book in result:
            self.assertGreaterEqual(book["published_year"], 2020)
            self.assertLessEqual(book["published_year"], 2021)

    def test_search_by_year_from_only(self):
        """只有 year_from"""
        result = search_books(self.conn, {"year_from": 2021})
        for book in result:
            self.assertGreaterEqual(book["published_year"], 2021)

    def test_search_combined_filters(self):
        """組合多個條件"""
        result = search_books(self.conn, {"author": "王", "year_from": 2021})
        self.assertEqual(len(result), 1)
        self.assertEqual(result[0]["title"], "Python 進階")

    def test_search_limit_default(self):
        """limit 預設值應該是 10"""
        # 新增更多資料來測試
        for i in range(20):
            self.conn.execute(
                "INSERT INTO books (title, author, isbn, published_year) VALUES (?, ?, ?, ?)",
                (f"Book {i}", "Author", f"ISBN-X{i}", 2020),
            )
        self.conn.commit()

        result = search_books(self.conn, {})
        self.assertLessEqual(len(result), 10)

    def test_search_limit_max_100(self):
        """limit 最大值應該是 100"""
        result = search_books(self.conn, {"limit": 999})
        self.assertLessEqual(len(result), 100)

    def test_search_offset(self):
        """offset 分頁功能"""
        result_page1 = search_books(self.conn, {"limit": 2, "offset": 0})
        result_page2 = search_books(self.conn, {"limit": 2, "offset": 2})

        self.assertEqual(len(result_page1), 2)
        self.assertEqual(len(result_page2), 2)
        # 確保兩頁內容不同
        self.assertNotEqual(result_page1[0]["id"], result_page2[0]["id"])

    def test_search_returns_dict_list(self):
        """回傳值應該是 list[dict]"""
        result = search_books(self.conn, {})
        self.assertIsInstance(result, list)
        if result:
            self.assertIsInstance(result[0], dict)
            self.assertIn("id", result[0])
            self.assertIn("title", result[0])

    def test_search_order_by_id(self):
        """結果應依 id 升冪排序"""
        result = search_books(self.conn, {})
        ids = [book["id"] for book in result]
        self.assertEqual(ids, sorted(ids))


# 執行測試
suite = unittest.TestLoader().loadTestsFromTestCase(TestSearchBooks)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

passed = result.testsRun - len(result.failures) - len(result.errors)
print(f"\n結果: {passed}/{result.testsRun} 測試通過")

---

### Part 1 完成檢查

執行下面的 cell 來檢查所有 function 的測試結果：

In [ ]:
from io import StringIO

print("=" * 60)
print("Part 1 測試總結")
print("=" * 60)

all_tests = [
    ("validate_book_data", TestValidateBookData),
    ("create_book", TestCreateBook),
    ("search_books", TestSearchBooks),
]

total_passed = 0
total_tests = 0

for name, test_class in all_tests:
    suite = unittest.TestLoader().loadTestsFromTestCase(test_class)
    runner = unittest.TextTestRunner(verbosity=0, stream=StringIO())

    try:
        result = runner.run(suite)
        passed = result.testsRun - len(result.failures) - len(result.errors)
        total_passed += passed
        total_tests += result.testsRun

        status = "PASS" if passed == result.testsRun else "FAIL"
        print(f"{name}: {passed}/{result.testsRun} [{status}]")

    except Exception as e:
        print(f"{name}: ERROR - {str(e)[:50]}")

print("=" * 60)
print(f"總計: {total_passed}/{total_tests} 測試通過")

if total_passed == total_tests:
    print("\n所有測試通過！")
else:
    print(f"\n還有 {total_tests - total_passed} 個測試需要修正")

---

## Part 2: Helicone API 分析

### 目標

使用 Helicone API 取回使用資料，分析 token 使用情況

---

In [ ]:
import pandas as pd
import time

# Helicone API endpoints
HELICONE_USER_API = "https://api.helicone.ai/v1/user/query"
HELICONE_REQUEST_API = "https://api.helicone.ai/v1/request/query-clickhouse"


def get_helicone_user_stats(user_id):
    """從 Helicone API 取得使用者聚合統計資料"""
    headers = {
        "Authorization": f"Bearer {HELICONE_API_KEY}",
        "Content-Type": "application/json",
    }

    query = {
        "userIds": [user_id],
    }

    response = requests.post(HELICONE_USER_API, headers=headers, json=query)

    if response.status_code in [200, 201]:
        return response.json()
    else:
        print(f"API 請求失敗: {response.status_code}")
        return None


def get_helicone_requests(user_id=None, limit=50):
    """從 Helicone API 取得個別請求記錄"""
    headers = {
        "Authorization": f"Bearer {HELICONE_API_KEY}",
        "Content-Type": "application/json",
    }

    # 建立 filter
    if user_id:
        query_filter = {
            "request_response_rmt": {
                "user_id": {
                    "equals": user_id
                }
            }
        }
    else:
        query_filter = "all"

    query = {
        "filter": query_filter,
        "offset": 0,
        "limit": limit,
        "sort": {
            "created_at": "desc"
        }
    }

    response = requests.post(HELICONE_REQUEST_API, headers=headers, json=query)

    if response.status_code in [200, 201]:
        return response.json()
    else:
        print(f"API 請求失敗: {response.status_code}")
        print(f"回應: {response.text}")
        return None


print("Helicone API 函式已準備")

In [ ]:
# 取得使用統計

print(f"正在取得 {USER_NAME} 的 Helicone 統計資料...\n")

stats = get_helicone_user_stats(USER_NAME)

if stats:
    print("API 回應:")
    print(json.dumps(stats, indent=2, ensure_ascii=False))
else:
    print("未取得資料，請確認 USER_NAME 設定正確")

In [ ]:
# 使用統計摘要

if stats and "data" in stats and len(stats["data"]) > 0:
    user_stats = stats["data"][0]

    print("=" * 50)
    print(f"使用者: {user_stats.get('user_id', USER_NAME)}")
    print("=" * 50)

    prompt_tokens = user_stats.get("prompt_tokens", 0)
    completion_tokens = user_stats.get("completion_tokens", 0)
    total_tokens = prompt_tokens + completion_tokens
    cost_usd = user_stats.get("cost", 0)
    request_count = user_stats.get("count", 0)

    print(f"\n總請求次數: {request_count}")
    print(f"總 Input Tokens: {prompt_tokens}")
    print(f"總 Output Tokens: {completion_tokens}")
    print(f"總 Tokens: {total_tokens}")
    print(f"總成本: ${cost_usd:.6f} USD")

else:
    print("沒有找到使用資料")
    print("可能原因：")
    print("1. USER_NAME 設定不正確")
    print("2. 尚未發送任何 API 請求")
    print("3. 時間範圍內沒有記錄")

### 個別請求記錄

除了聚合統計，我們也可以查看每一筆 API 請求的詳細資料：

---

## Lab 完成

### 這次你學會了

1. **Helicone 設定**: 透過 gateway 自動追蹤 API 使用
2. **使用 LLM 生成 function**: 提供 signature + docstring，讓 LLM 實作
3. **Unit Test 驗證**: 確保 LLM 生成的程式碼正確
4. **Helicone API 分析**: 程式化取得使用資料

### 關鍵學習

- 清楚的 function signature 和 docstring 能讓 LLM 生成更準確的實作
- Unit Test 是驗證 LLM 生成程式碼的有效方式
- Helicone 可以幫助追蹤和分析 API 使用成本

In [ ]:
# 取得個別請求記錄

from tabulate import tabulate

print(f"正在取得 {USER_NAME} 的個別請求記錄...\n")

request_data = get_helicone_requests(user_id=USER_NAME, limit=20)

if request_data and "data" in request_data and len(request_data["data"]) > 0:
    print(f"取得 {len(request_data['data'])} 筆記錄\n")

    records = []
    for req in request_data["data"]:
        # 取得 custom properties (如果有的話)
        properties = req.get("custom_properties", {}) or {}

        records.append({
            "時間": req.get("request_created_at", "")[:19].replace("T", " "),
            "模型": req.get("model", "N/A"),
            "Input": req.get("prompt_tokens", 0),
            "Output": req.get("completion_tokens", 0),
            "Total": req.get("total_tokens", 0),
            "成本 (USD)": f"${req.get('cost', 0):.6f}",
        })

    df = pd.DataFrame(records)
    print(tabulate(df, headers='keys', tablefmt='plain', showindex=False))
else:
    print("未找到個別請求記錄")
    print("\n提示：如果 filter 格式不對，可以嘗試不帶 user_id 查詢：")
    print("request_data = get_helicone_requests(limit=10)")

In [ ]:
# 依模型類型統計

if request_data and "data" in request_data and len(request_data["data"]) > 0:
    print("=" * 50)
    print("依模型類型統計")
    print("=" * 50)

    task_stats = df.groupby("模型").agg({
        "Total": "sum",
        "時間": "count"
    }).rename(columns={"Total": "總 Tokens", "時間": "請求次數"})

    print(task_stats.to_string())
else:
    print("沒有資料可供統計")